In [ ]:
!pip install -q ultralytics==8.3.221

In [ ]:
# Controlled speed benchmark: Faster R-CNN vs YOLOv8s
import time
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def measure(fn, warmup=10, iters=100, label=""):
    """Run fn() warmup times, then iters times; return mean ms and std."""
    # warmup
    for _ in range(warmup):
        fn()
    cuda_sync()

    # timed
    times = []
    for _ in range(iters):
        cuda_sync()
        t0 = time.perf_counter()
        fn()
        cuda_sync()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)   # ms

    times = np.array(times)
    mean = times.mean()
    std = times.std()
    print(f"  {label:<22} {mean:8.2f} ± {std:6.2f} ms   "
          f"({1000/mean:6.1f} it/s)")
    return mean, std

def benchmark_faster_rcnn(ckpt_path, images_pil, device, batch_size=4):
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    import torchvision.transforms as T

    print(f"\n[Faster R-CNN] batch={batch_size}, device={device}")

    # load model
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, 3)   # bg + smoke + fire
    model.transform.min_size = (640,)
    model.transform.max_size = 640
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()

    # precompute tensors (exclude PIL decode & ToTensor from timing)
    to_tensor = T.ToTensor()
    tensors = [to_tensor(im) for im in images_pil]
    # stack not possible (variable size), keep as list
    n_batches = len(tensors) // batch_size

    def run_one_batch(start):
        batch = [t.to(device, non_blocking=True)
                 for t in tensors[start:start + batch_size]]
        with torch.no_grad():
            _ = model(batch)

    # measure end-to-end (preprocess+forward+postprocess, GPU time only)
    def full():
        for b in range(n_batches):
            run_one_batch(b * batch_size)

    mean, std = measure(full, warmup=2, iters=10,
                        label=f"full pipeline ({n_batches} batches)")

    total_imgs = n_batches * batch_size
    fps = 1000.0 / mean * total_imgs
    print(f"  → throughput: {fps:.2f} img/s")
    return dict(model="Faster R-CNN", batch=batch_size,
                ms_per_batch=mean, ms_std=std,
                imgs_per_batch=batch_size,
                fps=fps)

def benchmark_yolo(ckpt_path, images_pil, device, batch_size=4,
                   protocol_dir=None):
    import sys
    if protocol_dir and protocol_dir not in sys.path:
        sys.path.insert(0, protocol_dir)
    from final_yolo_protocol import predict_with_baseline_input
    from ultralytics import YOLO

    print(f"\n[YOLOv8s] batch={batch_size}, device={device}")

    core = YOLO(ckpt_path).model.to(device).float().eval()

    n_batches = len(images_pil) // batch_size

    def run_one_batch(start):
        batch = images_pil[start:start + batch_size]   # PIL directly
        with torch.no_grad():
            _ = predict_with_baseline_input(
                core, batch, device=device,
                score_floor=0.001, nms_iou=0.7, max_det=100,
            )

    def full():
        for b in range(n_batches):
            run_one_batch(b * batch_size)

    mean, std = measure(full, warmup=2, iters=10,
                        label=f"full pipeline ({n_batches} batches)")

    total_imgs = n_batches * batch_size
    fps = 1000.0 / mean * total_imgs
    print(f"  → throughput: {fps:.2f} img/s")
    return dict(model="YOLOv8s", batch=batch_size,
                ms_per_batch=mean, ms_std=std,
                imgs_per_batch=batch_size,
                fps=fps)


# Run benchmark
# Paths
FRCN_CKPT  = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints/faster_rcnn_best.pth"
YOLO_CKPT  = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints/yolo_selected_best.pt"
PROTOCOL_DIR = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints"
TEST_IMAGES_DIR = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/images"

# Load 100 test images into PIL (once, in RAM)
img_paths = sorted(
    p for p in Path(TEST_IMAGES_DIR).iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)[:100]   # use first 100

print(f"Loading {len(img_paths)} test images into RAM...")
images_pil = []
for p in tqdm(img_paths):
    with Image.open(p) as im:
        images_pil.append(im.convert("RGB"))   # copy, so file closes
print(f"Loaded. First image size: {images_pil[0].size}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Run both, in the same session, back to back
results = []
results.append(benchmark_faster_rcnn(FRCN_CKPT, images_pil, DEVICE, batch_size=1))
results.append(benchmark_faster_rcnn(FRCN_CKPT, images_pil, DEVICE, batch_size=4))
results.append(benchmark_yolo(YOLO_CKPT, images_pil, DEVICE, batch_size=1,
                              protocol_dir=PROTOCOL_DIR))
results.append(benchmark_yolo(YOLO_CKPT, images_pil, DEVICE, batch_size=4,
                              protocol_dir=PROTOCOL_DIR))

# Summary table
print("\n" + "=" * 70)
print(f"{'Model':<16} {'batch':>6} {'ms/batch':>12} {'± std':>10} {'img/s':>10}")
print("-" * 70)
for r in results:
    print(f"{r['model']:<16} {r['batch']:>6} "
          f"{r['ms_per_batch']:>12.2f} {r['ms_std']:>10.2f} {r['fps']:>10.2f}")
print("=" * 70)

# Save
import json
Path("/kaggle/working/evaluation").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/evaluation/speed_benchmark.json").write_text(
    json.dumps(results, indent=2))
print("\nSaved → /kaggle/working/evaluation/speed_benchmark.json")